[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aelkharrim/FDM5-3P/blob/main/notebooks/07_pandas.ipynb)

# Python pour la physique — 7. Pandas : manipuler des tableaux de données

> 💡 **Astuce**
>
> **Rappel** : exécutez les cellules dans l'ordre avec **Maj + Entrée**. Les solutions des exercices sont masquées dans des cellules « Solution » : essayez d'abord par vous-même !

In [ ]:
# Cellule de préparation (facultative) : exécutez-la une fois au début.
# Sur Google Colab, NumPy, SciPy, Matplotlib et SymPy sont déjà installés.
import sys
EN_COLAB = 'google.colab' in sys.modules
print('Exécution dans Google Colab' if EN_COLAB else 'Exécution en local')

# Pourquoi pandas ?

Lors d'une séance de travaux pratiques, on obtient typiquement un **tableau de mesures** : chaque ligne est une mesure (ou une expérience), chaque colonne une grandeur (longueur, durée, température, nom de l'opérateur…). Un tableau NumPy convient bien aux données purement numériques et homogènes, mais il manque :

- de **noms de colonnes** ;
- de la possibilité de mélanger **nombres et texte** ;
- d'outils pour gérer les **valeurs manquantes** ;
- d'opérations de regroupement, de tri, de fusion…

**pandas** est le « tableur Python » : son objet principal, le `DataFrame`, est un tableau à colonnes nommées, avec de puissants mécanismes de sélection et de regroupement. Il lit et écrit les fichiers CSV et Excel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Créer un DataFrame

## À partir de listes ou de tableaux

Un `DataFrame` peut être vu comme un dictionnaire de « séries » 1D (listes ou tableaux NumPy). Voici des mesures de la période d'un pendule pour différentes longueurs :

In [ ]:
mesures = pd.DataFrame({
    'longueur_m': [0.20, 0.40, 0.60, 0.80, 1.00, 1.20],
    'periode_s':  [0.90, 1.27, 1.55, 1.79, 2.01, 2.20],
})
mesures

On peut aussi partir de tableaux NumPy :

In [ ]:
t = np.linspace(-6, 6, 20)
pd.DataFrame({'t': t, 'sin': np.sin(t), 'cos': np.cos(t)}).head()

## En lisant un fichier CSV

Dans la pratique, les données viennent d'un fichier : un export de logiciel d'acquisition, un tableur, un fichier de l'enseignant. Fabriquons d'abord un petit fichier CSV de TP à partir de données simulées, pour pouvoir le relire ensuite :

In [ ]:
rng = np.random.default_rng(12)
noms = ['Alice', 'Bruno', 'Chloé', 'David']
lignes = []
for operateur in noms:
    for L in [0.25, 0.50, 0.75, 1.00]:
        T = 2 * np.pi * np.sqrt(L / 9.81) * (1 + rng.normal(0, 0.02))
        lignes.append((operateur, L, round(T, 3), 0.1 if L < 0.7 else 0.2))
df_brut = pd.DataFrame(lignes, columns=['operateur', 'longueur_m', 'periode_s', 'masse_kg'])
df_brut.loc[5, 'periode_s'] = np.nan          # une mesure oubliée
df_brut.to_csv('pendule.csv', sep=';', index=False, decimal=',')
print(open('pendule.csv').read()[:300])

Ce fichier utilise le **point-virgule** comme séparateur et la **virgule** comme séparateur décimal (comme dans un tableur français). Lisons-le avec `read_csv` :

In [ ]:
df = pd.read_csv('pendule.csv', sep=';', decimal=',')
df

> 💡 **Astuce**
>
> **Avec Google Drive** : pour lire un fichier de votre Drive, montez-le d'abord :
>
> ```python
> from google.colab import drive
> drive.mount('/content/drive')
> df = pd.read_csv('/content/drive/MyDrive/mes_donnees/pendule.csv', sep=';', decimal=',')
> ```
>
> Pour un fichier Excel : `pd.read_excel('fichier.xlsx')`. Pour importer un fichier depuis votre ordinateur : `from google.colab import files; files.upload()`.



> ⚠️ **Attention**
>
> **Valeurs manquantes.** Une période est manquante dans le fichier (`NaN`, *Not a Number*). Si les valeurs manquantes sont codées autrement (par exemple `.` ou `-999`), indiquez-le avec `na_values=...` dans `read_csv`, sinon les statistiques seront fausses.

# Explorer les données

In [ ]:
df.shape          # (lignes, colonnes)

In [ ]:
df.columns        # noms des colonnes

In [ ]:
df.head()         # les premières lignes (df.tail() pour les dernières)

In [ ]:
df.info()         # types des colonnes et nombre de valeurs non manquantes

La méthode `describe` donne un résumé statistique rapide, pratique pour un gros tableau :

In [ ]:
df.describe()

# Sélectionner des données

**Une colonne**, par son nom (le résultat est une `Series`) :

In [ ]:
df['periode_s']

**Plusieurs colonnes** (le résultat est un `DataFrame`) :

In [ ]:
df[['operateur', 'periode_s']]

**Des lignes**, par position avec `iloc`, ou par étiquette avec `loc` :

In [ ]:
df.iloc[0:3]                   # trois premières lignes

In [ ]:
df.loc[0:3, ['longueur_m', 'periode_s']]   # attention : avec loc, la borne 3 est incluse

**Des lignes qui vérifient une condition** (filtre booléen, comme les masques NumPy) :

In [ ]:
df[df['operateur'] == 'Alice']

In [ ]:
df[(df['longueur_m'] >= 0.5) & (df['masse_kg'] == 0.2)]    # & = ET, | = OU, ~ = NON

Moyenne d'une sélection :

In [ ]:
df[df['operateur'] == 'Alice']['periode_s'].mean()

# Créer de nouvelles colonnes

Les opérations se font colonne par colonne, comme avec NumPy. Calculons la valeur de $g$ déduite de chaque mesure, avec $g = 4\pi^2 L / T^2$ :

In [ ]:
df['g'] = 4 * np.pi**2 * df['longueur_m'] / df['periode_s']**2
df.head()

Les fonctions NumPy s'appliquent directement aux colonnes :

In [ ]:
df['log_T'] = np.log10(df['periode_s'])
df[['periode_s', 'log_T']].head(3)

# Valeurs manquantes

In [ ]:
df.isna().sum()          # nombre de valeurs manquantes par colonne

In [ ]:
df[df['periode_s'].isna()]     # la ligne concernée

On peut **supprimer** les lignes incomplètes, ou **remplacer** les valeurs manquantes :

In [ ]:
df_propre = df.dropna()                  # supprime les lignes contenant un NaN
print(len(df), "->", len(df_propre))

In [ ]:
df['periode_s'].fillna(df['periode_s'].mean()).head(8)   # remplace par la moyenne (à utiliser avec prudence !)

> ⚠️ **Attention**
>
> Remplacer une mesure manquante par une moyenne **invente** une donnée. En physique, il vaut généralement mieux ignorer la mesure manquante. Notez que `mean()`, `std()`, etc. ignorent déjà les `NaN` par défaut.

# Regrouper et agréger : `groupby`

`groupby` découpe un tableau selon les valeurs d'une variable catégorielle (ici l'opérateur), puis calcule une statistique pour chaque groupe :

In [ ]:
df_propre = df.dropna()
df_propre.groupby('operateur')['g'].mean()

Plusieurs statistiques à la fois avec `agg` :

In [ ]:
df_propre.groupby('operateur')['g'].agg(['count', 'mean', 'std'])

On peut regrouper selon plusieurs colonnes :

In [ ]:
df_propre.groupby(['masse_kg', 'longueur_m'])['periode_s'].mean()

Et obtenir directement toutes les moyennes d'un tableau groupé :

In [ ]:
df_propre.groupby('operateur')[['periode_s', 'g']].mean()

> 📝 **Note**
>
> D'autres fonctions d'agrégation courantes : `median`, `min`, `max`, `sum`, `count` (pratique pour compter les valeurs manquantes dans chaque sous-ensemble).


Un **diagramme en boîtes** (*boxplot*) compare rapidement les distributions des groupes :

In [ ]:
df_propre.boxplot(column='g', by='operateur')
plt.suptitle("")
plt.ylabel("g déduit (m/s²)")
plt.show()

> **Exercice**
>
> 1. Quelle est la valeur moyenne de $g$ pour l'ensemble des mesures valides ?
> 2. Combien de mesures valides chaque opérateur a-t-il faites ?
> 3. Quelle est la période moyenne pour chaque longueur ?

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
print("g moyen :", df_propre['g'].mean())
print(df_propre.groupby('operateur')['g'].count())
print(df_propre.groupby('longueur_m')['periode_s'].mean())

# Trier, renommer, supprimer

In [ ]:
df_propre.sort_values('g', ascending=False).head(3)       # les 3 plus grandes valeurs de g

In [ ]:
df_propre.rename(columns={'periode_s': 'T (s)', 'longueur_m': 'L (m)'}).head(3)

In [ ]:
df_propre.drop(columns=['log_T']).head(3)

# Fusionner des tableaux

`merge` associe deux tableaux selon une colonne commune, comme une jointure de base de données. Par exemple, on ajoute à chaque mesure l'incertitude de l'instrument de chaque longueur :

In [ ]:
incertitudes = pd.DataFrame({'longueur_m': [0.25, 0.50, 0.75, 1.00],
                             'u_L_mm': [1, 1, 2, 2]})
fusion = df_propre.merge(incertitudes, on='longueur_m')
fusion[['operateur', 'longueur_m', 'u_L_mm']].head()

`pd.concat([df1, df2])` met deux tableaux l'un sous l'autre (par exemple les données de deux groupes de TP).

# Représenter graphiquement

Les `DataFrame` ont une méthode `plot` qui appelle Matplotlib :

In [ ]:
moyennes = df_propre.groupby('longueur_m')['periode_s'].mean()
moyennes.plot(marker='o', figsize=(5, 3))
plt.ylabel("T (s)")
plt.show()

In [ ]:
df_propre['g'].plot(kind='hist', bins=8, edgecolor='white', figsize=(5, 3))
plt.xlabel("g (m/s²)")
plt.show()

Pour des figures plus abouties, on extrait les colonnes et on utilise Matplotlib directement :

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
for operateur, groupe in df_propre.groupby('operateur'):
    ax.plot(groupe['longueur_m'], groupe['periode_s']**2, 'o', label=operateur)
ax.set_xlabel("L (m)")
ax.set_ylabel(r"$T^2$ (s²)")
ax.legend()
plt.show()

# Série temporelle : une courbe de refroidissement

Un autre cas typique : un fichier d'acquisition avec le temps en première colonne. Simulons le refroidissement d'un liquide (loi de Newton), enregistré toutes les 30 s :

In [ ]:
rng = np.random.default_rng(5)
temps = np.arange(0, 1800, 30)
T_amb, T0, tau = 21.0, 85.0, 600.0
T_mes = T_amb + (T0 - T_amb) * np.exp(-temps / tau) + rng.normal(0, 0.3, temps.size)

refroid = pd.DataFrame({'t_s': temps, 'T_C': T_mes.round(2)})
refroid.to_csv('refroidissement.csv', index=False)
refroid = pd.read_csv('refroidissement.csv')
refroid.head()

Ajoutons une colonne en minutes et calculons une moyenne glissante pour lisser le bruit :

In [ ]:
refroid['t_min'] = refroid['t_s'] / 60
refroid['T_lisse'] = refroid['T_C'].rolling(window=5, center=True).mean()

plt.figure(figsize=(6, 4))
plt.plot(refroid['t_min'], refroid['T_C'], '.', label="mesures")
plt.plot(refroid['t_min'], refroid['T_lisse'], '-', label="moyenne glissante")
plt.xlabel("t (min)"); plt.ylabel("T (°C)"); plt.legend()
plt.show()

# Pandas + SciPy : ajuster un modèle

Les colonnes d'un `DataFrame` sont directement utilisables avec `curve_fit` (carnet 5). Ajustons la loi de Newton $T(t) = T_\text{amb} + (T_0 - T_\text{amb})\,e^{-t/\tau}$ :

In [ ]:
from scipy.optimize import curve_fit

def newton(t, T_amb, delta, tau):
    return T_amb + delta * np.exp(-t / tau)

popt, pcov = curve_fit(newton, refroid['t_s'], refroid['T_C'], p0=[20, 60, 500])
incert = np.sqrt(np.diag(pcov))
print(f"T_amb = {popt[0]:.2f} ± {incert[0]:.2f} °C")
print(f"tau   = {popt[2]:.0f} ± {incert[2]:.0f} s")

refroid['T_modele'] = newton(refroid['t_s'], *popt)
refroid['residu'] = refroid['T_C'] - refroid['T_modele']
print("écart-type des résidus :", refroid['residu'].std().round(3), "°C")

# Enregistrer ses résultats

In [ ]:
resultats = df_propre.groupby('operateur')['g'].agg(['count', 'mean', 'std']).round(3)
resultats.to_csv('resultats_g.csv')
resultats

Pour Excel : `resultats.to_excel('resultats_g.xlsx')`. Pour récupérer un fichier créé dans Colab sur votre ordinateur :

```python
from google.colab import files
files.download('resultats_g.csv')
```

Ou, pour l'enregistrer dans Google Drive : `resultats.to_csv('/content/drive/MyDrive/resultats_g.csv')` (après avoir monté le Drive).

# Exercice de synthèse : la loi d'Ohm

Un groupe a mesuré l'intensité $I$ (en mA) pour plusieurs tensions $U$ (en V) aux bornes de trois résistances. Voici le tableau :

In [ ]:
rng = np.random.default_rng(21)
U = np.arange(1, 11)
donnees_ohm = pd.concat([
    pd.DataFrame({'resistance': nom, 'U_V': U,
                  'I_mA': 1000 * U / R + rng.normal(0, 0.15, U.size)})
    for nom, R in [('R1', 100.0), ('R2', 220.0), ('R3', 470.0)]
], ignore_index=True)
donnees_ohm.head()

1. Combien de mesures y a-t-il par résistance ?
2. Ajoutez une colonne `I_A` (intensité en ampères).
3. Pour **chaque résistance**, ajustez une droite $U = R\,I$ (utilisez `np.polyfit` sur `I_A` et `U_V`) et affichez la valeur de $R$ trouvée.
4. Tracez les trois séries de points et leurs droites ajustées sur un même graphique, avec une légende.

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
print(donnees_ohm.groupby('resistance').size())

donnees_ohm['I_A'] = donnees_ohm['I_mA'] / 1000

plt.figure(figsize=(6, 4))
for nom, groupe in donnees_ohm.groupby('resistance'):
    pente, ordonnee = np.polyfit(groupe['I_A'], groupe['U_V'], 1)
    print(f"{nom} : R = {pente:.1f} Ω")
    plt.plot(groupe['I_A'] * 1000, groupe['U_V'], 'o', label=f"{nom} (R = {pente:.0f} Ω)")
    plt.plot(groupe['I_A'] * 1000, pente * groupe['I_A'] + ordonnee, '-', color=plt.gca().lines[-1].get_color())
plt.xlabel("I (mA)"); plt.ylabel("U (V)"); plt.legend()
plt.show()

# Aide-mémoire

| Je veux…                         | Commande                                          |
| -------------------------------- | ------------------------------------------------- |
| lire un CSV                      | `pd.read_csv('f.csv', sep=';', decimal=',')`      |
| lire un fichier Excel            | `pd.read_excel('f.xlsx')`                         |
| aperçu / résumé                  | `df.head()`, `df.info()`, `df.describe()`         |
| une colonne / plusieurs colonnes | `df['a']` / `df[['a', 'b']]`                      |
| filtrer des lignes               | `df[df['a'] > 3]`                                 |
| nouvelle colonne                 | `df['c'] = df['a'] * df['b']`                     |
| valeurs manquantes               | `df.isna().sum()`, `df.dropna()`                  |
| statistiques par groupe          | `df.groupby('col')['a'].agg(['mean', 'std'])`     |
| trier                            | `df.sort_values('a')`                             |
| fusionner / empiler              | `df1.merge(df2, on='col')` / `pd.concat([df1, df2])` |
| enregistrer                      | `df.to_csv('f.csv', index=False)`                 |
| tracer                           | `df.plot(x='a', y='b')`                           |

# Pour aller plus loin

- La [documentation de pandas](https://pandas.pydata.org/docs/) et son guide « 10 minutes to pandas » (en anglais).
- **Seaborn** (`import seaborn as sns`) : graphiques statistiques élégants construits sur pandas et Matplotlib.
- **Statsmodels** : modèles linéaires, régressions et tests statistiques avec des tableaux pandas.

---

*Fin du carnet 7.*